# 7. Cartesian coordinates and thermochemical corrections

Collects, for every optimized structure, the block of thermochemical corrections
printed by Gaussian and the optimized Cartesian coordinates, and writes them to a
single text file for deposition.

| Output | Note |
|---|---|
| `qm/coords.txt` | **written to `qm/`, not to `output/`** |

`output/` is excluded from version control, so a file written there would not be part
of the deposited dataset. `qm/coords.txt` is the file referred to in the Supporting
Information and is tracked by git.

**Input:** `qm/qm_nbo_t6311++g/*.out`.

In [ ]:
import cclib

from pathlib import Path

# Resolve paths relative to the repository root, so that the notebook runs
# both from notebooks/ and from the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
QM = ROOT / "qm" / "qm_nbo_t6311++g"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

COORDS_PATH = ROOT / "qm" / "coords.txt"

out_files = sorted(QM.glob("*.out"))    # sorted, so the file content is reproducible
if not out_files:
    raise FileNotFoundError(
        f"No Gaussian output files found in {QM}. "
        "Place the .com/.out pairs of the ten training solvents there."
    )
print(f"{len(out_files)} Gaussian output files found")

## Extract one block per solvent

The thermochemistry section of a Gaussian output begins with the line
`Zero-point correction=` and continues for a further seven lines, covering the thermal
corrections to the energy, enthalpy and Gibbs free energy and the corresponding sums
with the electronic energy. `line[1:]` removes the single leading space that Gaussian
prints at the start of each line.

The coordinates are taken from the XYZ block generated by cclib, with its first two
lines (atom count and comment) removed.

In [ ]:
N_THERMO_LINES = 7      # lines following "Zero-point correction="


def parse_out_file(path):
    """Return the thermochemistry block and the optimized geometry of one calculation."""
    text = ""

    with open(path) as f:
        line = f.readline()
        while line:
            if "Zero-point correction" in line:
                text += line[1:]
                for _ in range(N_THERMO_LINES):
                    text += f.readline()[1:]
            line = f.readline()

    # cclib writes a standard XYZ block; drop the atom count and comment lines
    geometry = cclib.io.ccread(str(path)).writexyz().split("\n")[2:]
    return text + "\n".join(geometry)

## Write the collected data

The file is truncated before the loop. Opening it in append mode without truncating —
as an earlier version of this notebook did — silently duplicates the whole content
every time the cell is re-executed.

In [ ]:
blocks = []
for path in out_files:
    smiles = path.stem                   # keeps SMILES containing "." intact
    blocks.append(f"smiles: {smiles}\n{parse_out_file(path)}")
    print(f"parsed {smiles}")

# written in one go, which makes accidental duplication impossible
COORDS_PATH.write_text("\n\n".join(blocks) + "\n")

print(f"\n{len(blocks)} structures written to {COORDS_PATH.relative_to(ROOT)}")

## Check the result

A quick sanity check that the file contains one entry per calculation and no
duplicates.

In [ ]:
written = COORDS_PATH.read_text()
entries = [b.split("\n", 1)[0].strip() for b in written.split("smiles: ") if b.strip()]

print(f"entries in file : {len(entries)}")
print(f"unique SMILES   : {len(set(entries))}")
assert len(entries) == len(set(entries)), "duplicate entries found"

print("\n".join(entries))